In [1]:
import tensorflow as tf

In [1]:
import os
import numpy as np
from pandas import read_csv
from numpy import dstack
def get_values_around_max(series):
    """
    Returns the 20 values before and after the maximum value in the given Pandas Series.
    If there are not 20 values before or after the maximum, it will return None.
    """
    max_value = series.max()
    max_index = series.idxmax()

    start = max_index - 20
    end = max_index + 21

    if start < 0 or end > len(series):
        return None

    return series.iloc[start:end]
# load a single file as a numpy array
def load_file(filepath):
  fall = read_csv(filepath, header=0)
  t=np.sqrt(fall['accel_x_list']**2+fall['accel_y_list']**2+fall['accel_z_list']**2)
  return get_values_around_max(t)

# load a list of files and return as a 3d numpy array
def load_group(filenames, prefix=''):
  loaded = []
  for name in filenames:
    data = load_file(prefix + name)
    if data is None:
       print(name)
    else:
      loaded.append(data.tolist())
      #loaded = dstack(loaded)
  # stack group so that features are the 3rd dimension
  #loaded = dstack(loaded)
  return loaded

def get_files_in_dir(directory, file_pattern='accel.csv'):
    """
    Returns a list of all files in the given directory that match the file_pattern.
    """
    file_list = []
    print(directory)
    for filename in os.listdir(directory):
        if filename.endswith(file_pattern):
            file_list.append(os.path.join(directory, filename))
    return file_list

# load a dataset group, such as train or test
def load_dataset_group(group, prefix='C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/'):
#def load_dataset_group(group, prefix='/Users/song/Downloads/WEDA-FALL/dataset/10Hz/'):  
  filepath = prefix + group + '/'
  # load all 9 files as a single array
  filenames = get_files_in_dir(filepath, file_pattern='accel.csv')

  # load input data
  X = load_group(filenames)
  # load class output
  #y = load_file(prefix + group + '/y_'+group+'.txt')
  return X
trainX = load_dataset_group('F01')
#trainX = trainX.append(load_dataset_group('F02'))
trainX = np.concatenate((trainX, load_dataset_group('F02')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('F03')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('F04')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('F05')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('F06')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('F07')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('F08')), axis=0)
trainY = np.ones((len(trainX),1)) 
lenofone= len(trainX)
print("fall:"+str(lenofone))
trainX = np.concatenate((trainX, load_dataset_group('D01')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D02')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D03')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D04')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D05')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D06')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D07')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D08')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D09')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D10')), axis=0)
trainX = np.concatenate((trainX, load_dataset_group('D11')), axis=0)
trainY = np.concatenate((trainY, np.zeros((len(trainX)-lenofone, 1))), axis=0)
print(len(trainX))

C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F01/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F02/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F03/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F04/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F05/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F05/U08_R02_accel.csv
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F05/U11_R02_accel.csv
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F06/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F06/U05_R03_accel.csv
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F07/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F07/U08_R03_accel.csv
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F08/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F08/U06_R01_accel.csv
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F08/U14_R03_accel.csv
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/F08/U14_R04_accel.csv
fall:343
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/D01/
C:/Users/ssli/Downloads/WEDAFALL/dataset/10Hz/D01/U02_R0

In [2]:
trainY[804]

array([0.])

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(trainX, trainY, test_size=0.2, random_state=42)

In [6]:
X_train.shape,X_test.shape

((644, 41), (161, 41))

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
# Define the 1D CNN model
model = Sequential()
model.add(Conv1D(32, 3, activation='relu', input_shape=(41, 1)))
model.add(MaxPooling1D(2))
model.add(Conv1D(64, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train.reshape(-1, 41, 1), y_train, epochs=10, batch_size=32, validation_data=(X_test.reshape(-1, 41, 1), y_test))

# Evaluate the model
loss, accuracy = model.evaluate(X_test.reshape(-1, 41, 1), y_test)
print(f'Test loss: {loss:.4f}, Test accuracy: {accuracy:.4f}')

C:\Users\ssli\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6906 - loss: 0.6556 - val_accuracy: 0.8323 - val_loss: 0.3946
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8782 - loss: 0.3051 - val_accuracy: 0.8571 - val_loss: 0.3709
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8996 - loss: 0.2504 - val_accuracy: 0.8385 - val_loss: 0.3621
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8954 - loss: 0.2416 - val_accuracy: 0.8634 - val_loss: 0.3958
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9034 - loss: 0.2214 - val_accuracy: 0.8385 - val_loss: 0.4430
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9060 - loss: 0.2549 - val_accuracy: 0.8509 - val_loss: 0.3712
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9138 - loss: 0.2054 - val_accuracy: 0.8571 - val_loss: 0.3457
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9113 - loss: 0.2285 - val_accuracy: 0.8634 - val_loss